In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from netCDF4 import Dataset

from scintill_ai.io import get_magnetometer_data, get_solar_data, get_solar_wind_data
from scintill_ai.preprocess import get_solar_position
from var import START_DATE, END_DATE, DATA_IN, SMAG_YEARS, SMAG_STATIONS

## INTERMAGNET

In [ ]:
# df_kou = get_magnetometer_data(
#     Path(DATA_IN, 'KOU')
# ).loc[START_DATE:END_DATE, 'h']

# df_ttb = get_magnetometer_data(
#     Path(DATA_IN, 'TTB')
# ).loc[START_DATE:END_DATE, 'h']

In [ ]:
# df_mag = pd.merge(
#     left=df_kou,
#     right=df_ttb,
#     how='inner',
#     left_index=True,
#     right_index=True,
#     suffixes=['_kou', '_ttb']
# )

## SuperMAG

In [ ]:
dfs = {}

for yr_ in SMAG_YEARS:
    filepath = Path(DATA_IN, 'supermag', f'all_stations_all{yr_}.netcdf')
    dfs[yr_] = get_magnetometer_data(file_path=filepath, stations_list=SMAG_STATIONS)

df = pd.concat(dfs.values(), axis=0)
df['h_tmk'] = df['h_ttb'] - df['h_kou']

In [ ]:
# df_plt = df#.loc['2024-05-09':'2024-05-12']

# fig, (ax1, ax2) = plt.subplots(figsize=(21, 14), nrows=2, ncols=1, sharex=True)

# for i, stat_ in enumerate(SMAG_STATIONS):
#     ax = ax1 if i == 0 else ax2
#     ax.plot(df_plt.index, df_plt[f'h_{stat_.lower()}'], color='tab:blue')
    
#     [ax.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]

#     perc_na = df_plt[f'h_{stat_.lower()}'].isna().sum() / df_plt.shape[0]
#     ax.set_ylabel('Horizontal component [nT]')
#     ax.set_ylim(0)
#     ax.set_title(f'{stat_} magnetometer (NaN values: {perc_na:.1%})', fontsize=16, fontweight='bold')
#     ax.grid(True, axis='y', linewidth=0.3, alpha=0.8)

# plt.tight_layout()
# # plt.savefig('magnetometers.png', dpi=500)
# plt.show()

In [ ]:
df_plt = df.loc['2024-01-05':'2024-01-09']

fig, ax = plt.subplots(figsize=(21, 7))

ax.plot(df_plt.index, df_plt['h_tmk'], color='tab:blue')
[ax.spines[side].set_visible(False) for side in ['top', 'right', 'left', 'bottom']]

perc_na = df_plt['h_tmk'].isna().sum() / df_plt.shape[0]
ax.set_ylabel('Horizontal component [nT]')
# ax.set_ylim(0)
ax.set_title(f'TTB - KOU (NaN values: {perc_na:.1%})', fontsize=16, fontweight='bold')
ax.grid(True, axis='y', linewidth=0.3, alpha=0.8)

plt.tight_layout()
# plt.savefig('tmk_magnetometers.png', dpi=500)
plt.show()

## GFZ

In [ ]:
df_solar = get_solar_data(START_DATE, END_DATE)

## OMNIweb

In [ ]:
df_omni = get_solar_wind_data(Path(DATA_IN, 'omniweb')).loc[
    START_DATE:END_DATE,
    ['field_magnitude_avg', 'wind_speed', 'wind_density', 'wind_pressure', 'eletric_field']
]

## ISMR

$$ S_{4,\hspace{0.15 em}\mathrm{denoised}} = \mathrm{Re}\left( \sqrt{S_4^2 - S_{4,\hspace{0.15 em}\mathrm{noise}}^2} \right) $$

- for loop che suddivide l'anno in periodi di 3 mesi alla volta al più (magari in maniera asincrona), sputa fuori pickle appena finisce, poi unisce tutti i pezzi a formare l'anno, infine cancella i pickle intermedi quando ha finito

In [ ]:
from scintill_ai.io_async import get_aggregated_gnss_data_by_month_async

start = "2024-01-01"
end = "2024-02-29"
station_name = "PRU2"
fields = "time_utc, svid, azim, elev, s4, s4_correction, locktime_l1"
output_dir = f'../data/out/ismr/{station_name.lower()}'

df = await get_aggregated_gnss_data_by_month_async(start, end, station_name, fields, output_dir)

In [ ]:
pd.read_pickle('../data/out/ismr/pru2/2024_01.pickle')

## Solar Zenith Angle

In [ ]:
# get_solar_position(
#     df_XXX.index, columns=['zenith'], altitude=0,
# ).round(1)